# 02 — Exploración de producción

Punto de partida para analizar cada pozo. Ejecutar primero la revisión de datos. No se imputan faltantes ni se eliminan paradas.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Abrir desde la raíz del proyecto o notebooks/."
ARCHIVO = ROOT / "data" / "raw" / "Volve production data.xlsx"

# Los identificadores se conservan como texto.
diario = pd.read_excel(ARCHIVO, sheet_name="Daily Production Data", dtype={
    "NPD_WELL_BORE_CODE": "string", "NPD_FIELD_CODE": "string", "NPD_FACILITY_CODE": "string"
})
diario["DATEPRD"] = pd.to_datetime(diario["DATEPRD"], errors="raise")
mensual = pd.read_excel(ARCHIVO, sheet_name="Monthly Production Data", skiprows=[1],
                        na_values=["NULL"], dtype={"NPDCode": "string"})


## Cómo leer los nombres de las variables

Las tablas y los gráficos muestran nombres en español. En el código se conservan los nombres originales para poder rastrear cada variable hasta el Excel. Los códigos de pozos y los códigos `OP` y `WI` se conservan como identificadores; el significado de estos últimos sigue pendiente de confirmación.

La tabla de equivalencias permite relacionar ambas versiones. Las unidades y las interpretaciones pendientes no se dan por confirmadas al traducir.


In [ ]:
# Traducciones de presentación: los datos conservan sus nombres y valores originales.
etiquetas_es = {'DATEPRD': 'Fecha del registro',
 'WELL_BORE_CODE': 'Identificador textual del pozo',
 'NPD_WELL_BORE_CODE': 'Código NPD del pozo',
 'NPD_WELL_BORE_NAME': 'Nombre del pozo',
 'NPD_FIELD_CODE': 'Código del campo',
 'NPD_FIELD_NAME': 'Nombre del campo',
 'NPD_FACILITY_CODE': 'Código de instalación',
 'NPD_FACILITY_NAME': 'Nombre de instalación',
 'ON_STREAM_HRS': 'Horas operativas (h)',
 'AVG_DOWNHOLE_PRESSURE': 'Presión promedio de fondo',
 'AVG_DOWNHOLE_TEMPERATURE': 'Temperatura promedio de fondo',
 'AVG_DP_TUBING': 'Diferencia de presión promedio en la tubería',
 'AVG_ANNULUS_PRESS': 'Presión promedio del espacio anular',
 'AVG_CHOKE_SIZE_P': 'Apertura promedio de la válvula de restricción',
 'AVG_CHOKE_UOM': 'Unidad de apertura de la válvula',
 'AVG_WHP_P': 'Presión promedio en cabeza de pozo',
 'AVG_WHT_P': 'Temperatura promedio en cabeza de pozo',
 'DP_CHOKE_SIZE': 'Diferencia de presión en la válvula de restricción',
 'BORE_OIL_VOL': 'Volumen de petróleo (unidad pendiente)',
 'BORE_GAS_VOL': 'Volumen de gas (unidad pendiente)',
 'BORE_WAT_VOL': 'Volumen de agua producida (unidad pendiente)',
 'BORE_WI_VOL': 'Volumen de agua inyectada (interpretación y unidad pendientes)',
 'FLOW_KIND': 'Tipo de flujo',
 'WELL_TYPE': 'Tipo de pozo (código)',
 'Wellbore name': 'Nombre del pozo',
 'NPDCode': 'Código NPD del pozo',
 'Year': 'Año',
 'Month': 'Mes',
 'On Stream': 'Horas operativas del mes (h)',
 'Oil': 'Petróleo mensual (Sm³)',
 'Gas': 'Gas mensual (Sm³)',
 'Water': 'Agua producida mensual (Sm³)',
 'GI': 'Gas inyectado mensual (interpretación pendiente; Sm³)',
 'WI': 'Agua inyectada mensual (interpretación pendiente; Sm³)',
 'tipo': 'Tipo de dato',
 'valores_distintos': 'Valores distintos',
 'faltantes_total_pct': 'Faltantes en el total (%)',
 'faltantes_produccion_pct': 'Faltantes en producción (%)',
 'faltantes_inyeccion_pct': 'Faltantes en inyección (%)',
 'decision_vista': 'Decisión para la vista',
 'motivo': 'Motivo',
 'columna': 'Variable',
 'horas_cero': 'Registros con 0 horas operativas',
 'horas_fuera_0_24': 'Horas operativas fuera de 0 a 24',
 'petroleo_con_horas_cero': 'Petróleo positivo con 0 horas operativas',
 'production_con_tipo_WI': 'Flujo de producción con tipo de pozo WI',
 'registros': 'Registros',
 'count': 'Cantidad de datos disponibles',
 'mean': 'Promedio',
 'std': 'Desviación estándar',
 'min': 'Mínimo',
 '25%': 'Percentil 25',
 '50%': 'Mediana (percentil 50)',
 '75%': 'Percentil 75',
 'max': 'Máximo'}
tipos_es = {
    "object": "Texto o datos mixtos", "string": "Texto", "str": "Texto",
    "int64": "Número entero", "float64": "Número decimal",
    "datetime64[ns]": "Fecha", "datetime64[us]": "Fecha", "bool": "Verdadero o falso",
}

def tabla_en_espanol(tabla):
    """Crear una copia para mostrar, sin cambiar el dataframe de análisis."""
    vista = tabla.copy()
    vista = vista.rename(columns=etiquetas_es, index=etiquetas_es)
    if vista.index.name is not None:
        vista.index.name = etiquetas_es.get(vista.index.name, vista.index.name)
    if vista.columns.name is not None:
        vista.columns.name = etiquetas_es.get(vista.columns.name, vista.columns.name)
    if "Tipo de flujo" in vista.columns:
        vista["Tipo de flujo"] = vista["Tipo de flujo"].replace({
            "production": "Producción", "injection": "Inyección"
        })
    if "Tipo de dato" in vista.columns:
        vista["Tipo de dato"] = vista["Tipo de dato"].replace(tipos_es)
    return vista


In [ ]:
import plotly.express as px

produccion = diario.loc[diario["FLOW_KIND"].eq("production")].copy()
pozos = sorted(produccion["NPD_WELL_BORE_NAME"].unique())
print("Pozos con registros de producción:", pozos)

# Idioma de los textos y formato de fechas: explícitos, sin depender del navegador.
# Traducción local de los controles: también funciona sin conexión a Internet.
# Referencia: https://github.com/plotly/plotly.js/blob/master/lib/locales/es.js
idioma_plotly_es = {
    "dictionary": {
        "Download plot as a PNG": "Descargar gráfico como imagen PNG",
        "Download plot": "Descargar gráfico",
        "Zoom": "Ampliar una zona", "Pan": "Desplazar la vista",
        "Zoom in": "Acercar", "Zoom out": "Alejar",
        "Autoscale": "Ajustar escala a los datos", "Reset axes": "Restablecer ejes",
        "Reset view": "Restablecer vista", "Reset": "Restablecer",
        "Box Select": "Seleccionar con un rectángulo", "Lasso Select": "Seleccionar con un lazo",
        "Compare data on hover": "Comparar datos al pasar el cursor",
        "Show closest data on hover": "Mostrar el dato más cercano al cursor",
        "Toggle Spike Lines": "Mostrar u ocultar líneas guía",
        "Double-click to zoom back out": "Hacé doble clic para restablecer la vista",
        "Double-click on legend to isolate one trace": "Hacé doble clic en la leyenda para mostrar sólo esa serie",
        "Produced with Plotly.js": "Creado con Plotly.js",
        "Preparing image - this may take a few seconds": "Preparando imagen; puede tardar unos segundos",
        "Image download succeeded": "Imagen descargada correctamente",
        "Sorry, there was a problem downloading your image!": "No se pudo descargar la imagen",
    },
    "format": {
        "days": ["domingo", "lunes", "martes", "miércoles", "jueves", "viernes", "sábado"],
        "shortDays": ["dom", "lun", "mar", "mié", "jue", "vie", "sáb"],
        "months": ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"],
        "shortMonths": ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"],
        "date": "%d/%m/%Y", "decimal": ",", "thousands": ".",
    },
}
config_graficos = {
    "displaylogo": False, "scrollZoom": True, "responsive": True,
    "locale": "es", "locales": {"es": idioma_plotly_es},
}


## 1. ¿De qué fechas tenemos registros para cada pozo?

Este mapa muestra **únicamente el calendario y el flujo registrado**, no las horas ni los volúmenes. Cada barra agrupa días consecutivos del mismo tipo:

- **Azul:** existe un registro de producción.
- **Verde:** existe un registro de inyección.
- **Gris:** falta el registro diario entre la primera y la última fecha observadas del pozo.
- **Espacio en blanco:** fecha fuera del período observado de ese pozo. No sabemos qué ocurrió fuera de ese período.

Pasá el cursor sobre una barra: muestra pozo, primera y última fecha incluidas y cantidad de días. Podés ampliar una zona arrastrando, usar la rueda o el selector inferior. Un doble clic restablece la vista; la leyenda permite ocultar categorías.

Una barra azul o verde no confirma que el pozo haya operado: puede incluir días con cero horas. Eso se revisa por separado abajo. No interpretamos un día sin registro como producción cero ni como parada.


In [ ]:
assert not diario.duplicated(["NPD_WELL_BORE_CODE", "DATEPRD"]).any(), (
    "Revisar duplicados de pozo y fecha antes de construir el mapa."
)
tramos = []
for codigo_pozo, datos_pozo in diario.groupby("NPD_WELL_BORE_CODE", sort=True):
    datos_pozo = datos_pozo.sort_values("DATEPRD").set_index("DATEPRD")
    calendario = pd.date_range(datos_pozo.index.min(), datos_pozo.index.max(), freq="D")
    estado = pd.Series("Día sin registro", index=calendario)
    estado.loc[datos_pozo.index] = "Otro flujo o sin dato"
    estado.loc[datos_pozo.index[datos_pozo["FLOW_KIND"].eq("production")]] = "Registro de producción"
    estado.loc[datos_pozo.index[datos_pozo["FLOW_KIND"].eq("injection")]] = "Registro de inyección"
    grupos = estado.ne(estado.shift()).cumsum()
    nombre_pozo = " / ".join(sorted(datos_pozo["NPD_WELL_BORE_NAME"].dropna().unique()))
    for _, tramo in estado.groupby(grupos):
        tramos.append({
            "Pozo": nombre_pozo, "Código del pozo": codigo_pozo,
            "Desde": tramo.index.min(), "Hasta": tramo.index.max(),
            # Plotly usa un extremo final exclusivo; Hasta es la última fecha incluida.
            "Fin de barra": tramo.index.max() + pd.Timedelta(days=1),
            "Días": len(tramo), "Registro": tramo.iloc[0],
        })
tramos_calendario = pd.DataFrame(tramos).sort_values(["Pozo", "Desde"])
fig_cobertura = px.timeline(
    tramos_calendario, x_start="Desde", x_end="Fin de barra", y="Pozo", color="Registro",
    category_orders={"Pozo": sorted(tramos_calendario["Pozo"].unique())},
    color_discrete_map={"Registro de producción": "#2463a6", "Registro de inyección": "#168577",
                        "Día sin registro": "#b5b5b5", "Otro flujo o sin dato": "#333333"},
    custom_data=["Pozo", "Desde", "Hasta", "Días", "Registro"],
    title="Calendario de registros por pozo — ¿dónde tenemos datos?",
    template="plotly_white", height=560,
)
fig_cobertura.update_traces(hovertemplate=(
    "<b>Pozo %{customdata[0]}</b><br>%{customdata[4]}<br>"
    "Desde: %{customdata[1]|%d/%m/%Y}<br>Hasta: %{customdata[2]|%d/%m/%Y}<br>"
    "Días incluidos: %{customdata[3]}<extra></extra>"
))
fig_cobertura.update_yaxes(autorange="reversed", title="Pozo", categoryorder="array",
                            categoryarray=sorted(tramos_calendario["Pozo"].unique()))
fig_cobertura.update_xaxes(title="Fecha", tickformat="%Y-%m", rangeslider_visible=True,
                          rangeslider_thickness=0.08)
fig_cobertura.update_layout(legend={"title": "Tipo de registro", "orientation": "h",
                                    "y": 1.03, "x": 0}, margin={"t": 110, "b": 70})
fig_cobertura.show(config=config_graficos)


## 2. ¿Qué significa un día con 0 horas operativas?

`ON_STREAM_HRS` indica cuántas horas de operación declara el registro de ese día. **0 horas no significa medianoche:** significa que el archivo registra cero tiempo de operación. Por ejemplo, 24 h representa 24 horas registradas y 6 h representa seis horas.

Un cero puede ser compatible con un pozo detenido, pero no confirma la causa ni garantiza que el volumen registrado sea cero. Debemos revisar ambas variables. Un valor vacío significa que desconocemos las horas; no se convierte en cero.

El gráfico siguiente cuenta fechas distintas con **0 horas registradas**, considerando producción e inyección. Pasá el cursor para ver también días con registro, horas faltantes y porcentaje con cero horas. Los conteos dependen de cuánto tiempo cubre cada pozo: no sirven por sí solos para decir cuál funciona mejor. Los días sin registro no entran en ese porcentaje.


In [ ]:
filas_horas = []
for codigo_pozo, datos_pozo in diario.groupby("NPD_WELL_BORE_CODE", sort=True):
    dias_registrados = datos_pozo["DATEPRD"].nunique()
    dias_cero = datos_pozo.loc[datos_pozo["ON_STREAM_HRS"].eq(0), "DATEPRD"].nunique()
    filas_horas.append({
        "Pozo": " / ".join(sorted(datos_pozo["NPD_WELL_BORE_NAME"].dropna().unique())),
        "Días con 0 horas registradas": dias_cero,
        "Días con registro": dias_registrados,
        "Días con horas faltantes": datos_pozo.loc[datos_pozo["ON_STREAM_HRS"].isna(), "DATEPRD"].nunique(),
        "Porcentaje con 0 horas": round(100 * dias_cero / dias_registrados, 2),
    })
resumen_horas = pd.DataFrame(filas_horas).sort_values("Pozo")
fig_horas_cero = px.bar(
    resumen_horas, x="Días con 0 horas registradas", y="Pozo", orientation="h",
    text="Días con 0 horas registradas", color_discrete_sequence=["#e58c25"],
    custom_data=["Pozo", "Días con 0 horas registradas", "Días con registro",
                 "Días con horas faltantes", "Porcentaje con 0 horas"],
    title="Días en que el archivo registra 0 horas de operación", template="plotly_white", height=440,
)
fig_horas_cero.update_traces(textposition="outside", cliponaxis=False, hovertemplate=(
    "<b>Pozo %{customdata[0]}</b><br>Días con 0 horas: %{customdata[1]}<br>"
    "Días con registro: %{customdata[2]}<br>Días con horas faltantes: %{customdata[3]}<br>"
    "Porcentaje con 0 horas: %{customdata[4]:.2f}%<extra></extra>"
))
fig_horas_cero.update_yaxes(autorange="reversed")
fig_horas_cero.update_layout(margin={"r": 65})
fig_horas_cero.show(config=config_graficos)


## Seleccionar un pozo

In [ ]:
POZO = "15/9-F-12"
serie = (produccion.loc[produccion["NPD_WELL_BORE_NAME"].eq(POZO)]
         .sort_values("DATEPRD").set_index("DATEPRD"))
assert not serie.empty, "Seleccionar un pozo con registros de producción."
assert serie.index.is_unique, "Revisar duplicados antes de graficar."
# Reindexar sólo para el gráfico: los huecos quedan vacíos, sin interpolación.
serie_calendario = serie.reindex(pd.date_range(serie.index.min(), serie.index.max(), freq="D"))
display(tabla_en_espanol(serie[["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "ON_STREAM_HRS"]].describe()))

## 3. Evolución temporal del pozo seleccionado

Cada gráfico responde una pregunta: cuánto petróleo se registró, cuánta agua producida se registró y cuántas horas operativas se informaron. Los volúmenes diarios conservan su unidad pendiente de confirmación. Las escalas son diferentes; no compares alturas entre gráficos.

Pasá el cursor para consultar fecha, valor, flujo y horas registradas. Usá el selector inferior para acercarte a un período. En el gráfico de horas, los puntos naranjas marcan exactamente los registros con **0 h**, no los días sin dato.

Las líneas se interrumpen cuando falta el valor, por ausencia del registro o de la medición. No se interpolan ni se suavizan. El calendario anterior permite localizar días sin registro; el estado de la medición y las horas también se conserva en la tabla `datos_grafico`.


In [ ]:
datos_grafico = serie_calendario.rename_axis("Fecha").reset_index().copy()
datos_grafico["Hay registro"] = datos_grafico["Fecha"].isin(serie.index)
datos_grafico["Flujo"] = datos_grafico["FLOW_KIND"].map({"production": "Producción", "injection": "Inyección"}).fillna("Sin dato")
datos_grafico["Horas registradas"] = datos_grafico["ON_STREAM_HRS"].map(
    lambda h: "Sin dato" if pd.isna(h) else f"{h:.2f} h".replace(".", ",")
)
especificaciones = [
    ("BORE_OIL_VOL", "¿Cuánto petróleo se registró?", "Volumen de petróleo (unidad pendiente)", "#2463a6"),
    ("BORE_WAT_VOL", "¿Cuánta agua producida se registró?", "Volumen de agua producida (unidad pendiente)", "#168577"),
    ("ON_STREAM_HRS", "¿Cuántas horas operativas se registraron?", "Horas operativas (h)", "#9057b2"),
]
figuras_evolucion = {}
for columna, pregunta, etiqueta, color in especificaciones:
    fig = px.line(
        datos_grafico, x="Fecha", y=columna, labels={columna: etiqueta},
        custom_data=["Flujo", "Horas registradas"],
        title=f"{pregunta} — Pozo {POZO}", template="plotly_white", height=400,
        color_discrete_sequence=[color], render_mode="svg",
    )
    fig.update_traces(connectgaps=False, hovertemplate=(
        "Fecha: %{x|%d/%m/%Y}<br>" + etiqueta + ": %{y:.2f}<br>"
        "Flujo: %{customdata[0]}<br>Horas registradas: %{customdata[1]}<extra></extra>"
    ))
    if columna == "ON_STREAM_HRS":
        ceros = datos_grafico.loc[datos_grafico[columna].eq(0)]
        puntos_cero = px.scatter(ceros, x="Fecha", y=columna)
        for traza in puntos_cero.data:
            traza.update(name="0 horas registradas", showlegend=True,
                         marker={"color": "#e58c25", "size": 7},
                         hovertemplate="Fecha: %{x|%d/%m/%Y}<br>0 horas de operación registradas<extra></extra>")
            fig.add_trace(traza)
    fig.update_xaxes(title="Fecha", tickformat="%Y-%m", rangeslider_visible=True,
                     rangeslider_thickness=0.12)
    fig.update_layout(hovermode="x", legend={"orientation": "h", "y": 1.05, "x": 0})
    figuras_evolucion[columna] = fig
    fig.show(config=config_graficos)


## Preguntas para continuar

- ¿Cuándo cambia el comportamiento productivo?
- ¿Qué relación se observa con las horas operativas?
- ¿Qué sensores tienen información suficiente en cada período?
- ¿Qué diferencias aparecen al seleccionar otro pozo?

Registrar asociaciones sin interpretarlas como efectos causales.

## Conclusiones

Completar después de explorar los pozos.

## Copia interactiva para abrir en el navegador

Esta celda reúne los gráficos en un HTML local con las mismas explicaciones. El archivo se guarda en `reports/02_exploracion_interactiva.html`, fuera del control de versiones. No necesita conexión a Internet ni sube datos a un servicio. Para cambiar el pozo, modificá `POZO` arriba y ejecutá de nuevo sus celdas y esta exportación.


In [ ]:
from html import escape

graficos_exportar = [fig_cobertura, fig_horas_cero, *figuras_evolucion.values()]
encabezado_html = f"""<!doctype html><html lang="es"><head><meta charset="utf-8">
<title>WELLWISE-X — Exploración interactiva</title>
<style>body{{font-family:Arial,sans-serif;max-width:1400px;margin:30px auto;padding:0 20px;line-height:1.5}}</style>
</head><body><h1>Exploración de los registros de Volve</h1>
<p>Calendario de todos los pozos y evolución del pozo <strong>{escape(POZO)}</strong>.</p>
<p>Pasá el cursor para consultar valores; arrastrá o usá el selector inferior para ampliar fechas.
Un doble clic restablece la vista. La leyenda permite ocultar categorías.</p>
"""
explicaciones_html = [
    "<h2>1. Calendario: ¿de qué fechas tenemos registros?</h2><p>Azul: producción; verde: inyección; gris: día sin registro dentro del período observado. Los espacios blancos están fuera de ese período. Los colores no indican si el pozo operó. El cursor muestra fechas incluidas y duración.</p>",
    "<h2>2. Días con 0 horas de operación registradas</h2><p>0 horas no significa medianoche: la variable de horas operativas registra cero tiempo de operación para ese día. No confirma la causa ni garantiza volumen cero. Un dato faltante significa que desconocemos las horas. El porcentaje del cursor usa sólo días con registro; los conteos dependen del período observado de cada pozo.</p>",
    "<h2>3. Petróleo del pozo seleccionado</h2><p>Volumen diario en la unidad registrada, pendiente de confirmar. Las interrupciones de la línea indican valores ausentes, no ceros.</p>",
    "<h2>4. Agua producida del pozo seleccionado</h2><p>Es agua que sale del pozo, diferente del agua inyectada. Unidad diaria pendiente de confirmar. No compares alturas con el gráfico de petróleo: las escalas son distintas.</p>",
    "<h2>5. Horas de operación del pozo seleccionado</h2><p>Los puntos naranjas marcan registros de 0 h. Los días sin registro o sin medición quedan vacíos. No se interpolan ni se suavizan las curvas.</p>",
]
fragmentos_html = [encabezado_html]
for i, (grafico, explicacion) in enumerate(zip(graficos_exportar, explicaciones_html)):
    fragmentos_html.append(explicacion)
    fragmentos_html.append(grafico.to_html(full_html=False, include_plotlyjs=(i == 0), config=config_graficos))
fragmentos_html.append("</body></html>")
carpeta_reportes = ROOT / "reports"
carpeta_reportes.mkdir(exist_ok=True)
archivo_interactivo = carpeta_reportes / "02_exploracion_interactiva.html"
archivo_interactivo.write_text("\n".join(fragmentos_html), encoding="utf-8")
print("Copia interactiva guardada en:", archivo_interactivo)
